# GPU Wavelets (JPEG2000-style MRA) — Colab driver

Runtime → Change runtime type → **GPU** (T4 is fine). Then run cells top to bottom.

This builds the project with CMake and runs the deterministic 1D/2D/MRA tests.

In [12]:
!nvidia-smi
!nvcc --version | tail -1
!cmake --version | head -1

Fri Jul 31 17:48:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Get the code onto Colab
Clones the GitHub repo to `/content/wavelets`. Re-running pulls the latest.

In [23]:
# Clone the repo onto the Colab machine (re-runnable: pulls if already cloned).
TARGET = '/content/wavelets'
REPO = 'https://github.com/DivyeshJayswal/wavelets.git'
import os
if os.path.exists(TARGET):
    !cd {TARGET} && git pull --ff-only
else:
    !git clone {REPO} {TARGET}
%cd /content/wavelets
!ls

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 7 (delta 6), reused 7 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 1.59 KiB | 233.00 KiB/s, done.
From https://github.com/DivyeshJayswal/wavelets
   b8bfc53..39b6593  main       -> origin/main
Updating b8bfc53..39b6593
Fast-forward
 report.md         | 18 +++++++++++++++---
 src/wavelet3d.cuh |  4 ++++
 tests/test_3d.cu  |  5 ++++-
 3 files changed, 23 insertions(+), 4 deletions(-)
/content/wavelets
assets	CMakeLists.txt	       report.md  third_party		Wavelets.pdf
bench	common_guidelines.tex  src	  wavelets		wavelets.tex
build	README.md	       tests	  wavelets_colab.ipynb


In [24]:
import subprocess
cc = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader']
).decode().strip().split('\n')[0].replace('.', '')
print('compute capability =', cc)
!cmake -B build -DCMAKE_CUDA_ARCHITECTURES={cc} -DCMAKE_BUILD_TYPE=Release
!cmake --build build -j

compute capability = 75
-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /content/wavelets/build
[ 12%] Built target test_io
[ 25%] Built target test_1d
[ 37%] Built target test_fp16
[ 50%] Built target wavelet
[ 56%] Building CUDA object CMakeFiles/test_3d.dir/tests/test_3d.cu.o
[ 68%] Built target test_2d
[ 81%] Built target bench
[ 93%] Built target test_access
ptxas info    : 0 bytes gmem
ptxas info    : Compiling entry function '_Z13k2_interleaveIfEvPKT_PS0_iiii' for 'sm_75'
ptxas info    : Function properties for _Z13k2_interleaveIfEvPKT_PS0_iiii
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 15 registers, used 0 barriers, 384 bytes cmem[0]
ptxas info    : Compile time = 2.661 ms
ptxas info    : Compiling entry function '_Z15k2_inv_predict1IfEvPT_iiii' for 'sm_75'
ptxas info    : Function properties for _Z15k2_inv_predict1IfEvPT_iiii
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads

## Run tests (items 1, 2, 3, 4, 8, 9, 10, 14)
Deterministic binaries — `test_1d/2d/access/3d/fp16/io`. Each returns nonzero on failure; `ctest` aggregates.

In [25]:
!cd build && ctest --output-on-failure

Test project /content/wavelets/build
    Start 1: test_1d
1/6 Test #1: test_1d ..........................   Passed    0.96 sec
    Start 2: test_2d
2/6 Test #2: test_2d ..........................   Passed    0.34 sec
    Start 3: test_access
3/6 Test #3: test_access ......................   Passed    0.29 sec
    Start 4: test_3d
4/6 Test #4: test_3d ..........................   Passed    0.26 sec
    Start 5: test_fp16
5/6 Test #5: test_fp16 ........................   Passed    0.24 sec
    Start 6: test_io
6/6 Test #6: test_io ..........................   Passed    0.00 sec

100% tests passed, 0 tests failed out of 6

Total Test time (real) =   2.10 sec


## CLI demo — image I/O + round-trip (items 4, 13)
Reconstructs the castle image (round-trip) and writes the MRA representation.

In [ ]:
# Write our own outputs into assets/ so the report figure uses real project output.
!cd build && ./wavelet --input ../assets/Castle_Lichtenstein.jpg --output ../assets/recon.png --levels 3
!cd build && ./wavelet --input ../assets/Castle_Lichtenstein.jpg --output ../assets/mra.png --levels 3 --forward
from IPython.display import Image, display
display(Image('assets/recon.png'), Image('assets/mra.png'))

## Benchmarks — items 5, 6, 7, 10, 11
Saves markdown tables to `bench_results.md`; paste them into `report.md`'s `FILL` slots.

In [27]:
!cd build && ./bench | tee ../bench_results.md

# Benchmark results

GPU: Tesla T4 (sm_75), warmup=5 reps=30

### MRA forward throughput (fp32, levels=1)

| Size | time (ms) | Melem/s |
|---|---|---|
| 256² | 0.069 | 956 |
| 512² | 0.180 | 1453 |
| 1024² | 0.668 | 1571 |
| 2048² | 2.684 | 1563 |
| 4096² | 10.326 | 1625 |

### fp32 vs fp16 MRA forward (levels=3)

| Size | fp32 (ms) | fp16 (ms) | speedup |
|---|---|---|---|
| 512² | 0.213 | 0.223 | 0.96x |
| 1024² | 0.659 | 0.606 | 1.09x |
| 2048² | 2.941 | 2.687 | 1.09x |
| 4096² | 13.247 | 10.628 | 1.25x |

### Column pass: naive strided vs coalesced transpose (fp32, item 6)

_naive vs transpose max diff at 1024² = 0.00e+00 (match)_

| Size | naive (ms) | naive GB/s | transpose (ms) | transp GB/s | speedup |
|---|---|---|---|---|---|
| 512² | 0.103 | 20.4 | 0.029 | 71.4 | 3.50x |
| 1024² | 0.374 | 22.4 | 0.189 | 44.4 | 1.98x |
| 2048² | 1.696 | 19.8 | 0.888 | 37.8 | 1.91x |
| 4096² | 8.075 | 16.6 | 3.556 | 37.7 | 2.27x |

### Row pass: step-per-launch vs shared-memory tile (fp32, it

## Register / shared-memory usage (item 11)
`--ptxas-options=-v` is on, so per-kernel register/smem usage prints at compile time. Force a recompile of `bench` to surface it:

In [28]:
!cd build && touch ../bench/bench.cu && cmake --build . --target bench 2>&1 | grep -iE "ptxas info|registers|bytes smem|bytes stack"

ptxas info    : 0 bytes gmem
ptxas info    : Compiling entry function '_Z20k_row_forward_sharedIfEvPT_ii' for 'sm_75'
ptxas info    : Function properties for _Z20k_row_forward_sharedIfEvPT_ii
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 18 registers, used 1 barriers, 368 bytes cmem[0]
ptxas info    : Compile time = 5.931 ms
ptxas info    : Compiling entry function '_Z7k2_copyIfEvPKT_PS0_iiii' for 'sm_75'
ptxas info    : Function properties for _Z7k2_copyIfEvPKT_PS0_iiii
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 8 registers, used 0 barriers, 384 bytes cmem[0]
ptxas info    : Compile time = 1.216 ms
ptxas info    : Compiling entry function '_Z15k2_fwd_predict2IfEvPT_iiii' for 'sm_75'
ptxas info    : Function properties for _Z15k2_fwd_predict2IfEvPT_iiii
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 14 registers, used 0 barriers, 376 bytes cmem[0]
ptxas info  